# 03 — Detector Agent Training (PPO)
Trains the binary threat detector with attention-enhanced PPO policy on `NetworkEnv`.

In [ ]:
import sys; sys.path.insert(0,'..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
Path('../models').mkdir(exist_ok=True)
Path('../reports').mkdir(exist_ok=True)

## 1. Load Preprocessed Data

In [ ]:
from data.preprocess import load_and_preprocess
data=load_and_preprocess('../data/raw/cicids2017/synthetic_cicids.parquet')
X_train,y_train=data['X_train'],data['y_train']
X_val,y_val=data['X_val'],data['y_val']
X_test,y_test=data['X_test'],data['y_test']
print(f'Train:{X_train.shape} Val:{X_val.shape} Test:{X_test.shape}')

## 2. Inspect Custom Gym Environment

In [ ]:
from environment.network_env import NetworkEnv
env=NetworkEnv(X_train,y_train,max_steps=200,window_size=5)
obs,info=env.reset(seed=42)
print(f'Observation space: {env.observation_space}')
print(f'Action space:      {env.action_space}')
print(f'Obs shape:         {obs.shape} | dtype: {obs.dtype}')
print(f'Sample info:       {info}')
for step in range(5):
    action=env.action_space.sample()
    obs,reward,done,_,info=env.step(action)
    print(f'  Step {step+1}: action={action} reward={reward:.2f} f1={info["f1"]:.3f}')

## 3. Reward Shaping Sanity Check

In [ ]:
env=NetworkEnv(X_train,y_train,max_steps=500,window_size=5,render_mode='human')
env.reset(seed=0)
total_r=0
for i in range(20):
    r=env.step(1)[1]; total_r+=r
    env.render() if i%5==0 else None
print(f'\nTotal reward (always predict threat): {total_r:.2f}')

## 4. Train PPO Detector Agent

In [ ]:
from agents.detector_agent import train_detector
# Reduced timesteps for notebook demo — use 500_000 for full training
model=train_detector(
    X_train,y_train,X_val,y_val,
    total_timesteps=50_000,   # increase to 500_000 for production
    save_path='../models/detector',
    n_envs=2,
)
print('Training complete ✓')

## 5. Evaluate on Test Set

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from environment.network_env import NetworkEnv
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import seaborn as sns

env=NetworkEnv(X_test,y_test,max_steps=len(X_test),window_size=5)
obs,_=env.reset(seed=99)
y_true_bin,y_pred_bin=[],[]
done=False
while not done:
    action,_=model.predict(obs,deterministic=True)
    obs,r,done,_,info=env.step(int(action))
    y_true_bin.append(1 if y_test[env._step%len(y_test)]!=0 else 0)
    y_pred_bin.append(int(action))
f1=f1_score(y_true_bin[:len(y_pred_bin)],y_pred_bin,average='binary')
print(f'Detector F1 (binary): {f1:.4f}')
print(f'Final episode info: {info}')

## 6. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
cm=confusion_matrix(y_true_bin[:len(y_pred_bin)],y_pred_bin)
fig,ax=plt.subplots(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
            xticklabels=['Pred Benign','Pred Threat'],
            yticklabels=['True Benign','True Threat'],ax=ax)
ax.set_title('Detector Agent Confusion Matrix',fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/03_confusion.png',bbox_inches='tight')
plt.show()
fp_rate=cm[0,1]/(cm[0,0]+cm[0,1]+1e-9)
print(f'False Positive Rate: {fp_rate:.3f}  (baseline RF: ~0.042)')

## ✅ Detector Agent Summary
- **AttentionFlowExtractor**: sliding-window self-attention over flow history — novel architecture
- Asymmetric reward (FN_COST=2× FP_COST) reflects SOC alert fatigue economics
- Time bonus incentivises early detection in first 20% of episode
- Target: F1 > 95%, FPR < 2% (beats RF baseline of 4.2%)